# House Prices Competition - Modeling Notebook

**Objective:** Build and evaluate regression models to predict `SalePrice` (log-transformed)

**Metrics:** RMSE (on log scale) → RMSLE on original scale

**Approach:** Ridge baseline → XGBoost → LightGBM → Ensemble

**Validation:** 5-fold KFold with target encoding inside CV (no leakage)

## 1. Setup & Imports

In [68]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Modeling
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.preprocessing import LabelEncoder

# Advanced models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Hyperparameter tuning
import optuna
from optuna.samplers import TPESampler

# Misc
import joblib
from datetime import datetime

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Processed Data

In [69]:
# Load datasets
X_train = pd.read_csv("../data/X_train_processed.csv")
y_train = pd.read_csv("../data/y_train_processed.csv").values.ravel()  # Log-transformed SalePrice
X_test = pd.read_csv("../data/X_test_processed.csv")

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\ny_train stats - mean: {y_train.mean():.4f}, std: {y_train.std():.4f}")

# %%
# Quick sanity check - no missing values should remain
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

X_train shape: (1460, 201)
y_train shape: (1460,)
X_test shape: (1459, 201)

y_train stats - mean: 12.0241, std: 0.3993
Missing values in X_train: 0
Missing values in X_test: 0


## 3. Identify Columns for Target Encoding

In [70]:
# Columns that need target encoding (high cardinality)
target_encode_cols = ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']

# Verify they exist
for col in target_encode_cols:
    if col in X_train.columns:
        print(f"✓ {col} found - {X_train[col].nunique()} unique values")
    else:
        print(f"✗ {col} NOT found - check preprocessing")

✓ MSSubClass found - 15 unique values
✓ Neighborhood found - 25 unique values
✓ Exterior1st found - 15 unique values
✓ Exterior2nd found - 16 unique values


## 4. Simple Target Encoding Function (No Leakage)

For each fold: compute means on training fold, apply to validation fold

In [71]:
def apply_target_encoding(X_train_fold, y_train_fold, X_val_fold, columns, global_mean, smoothing=20):
    """
    Apply target encoding to validation fold based on training fold means.
    
    Returns:
        X_train_fold_encoded, X_val_fold_encoded (with encoded columns added)
    """
    X_train_enc = X_train_fold.copy()
    X_val_enc = X_val_fold.copy()
    
    if isinstance(y_train_fold, np.ndarray):
        y_train_fold = pd.Series(y_train_fold, index=X_train_fold.index)

    for col in columns:
        # Compute group means on training fold
        group_means = y_train_fold.groupby(X_train_fold[col]).agg(['mean', 'count'])
        
        # Apply smoothing: (mean * count + global_mean * smoothing) / (count + smoothing)
        smoothed = (group_means['mean'] * group_means['count'] + global_mean * smoothing) / (group_means['count'] + smoothing)
        
        # Create encoded column name
        encoded_col = f'{col}_target_enc'
        
        # Add to validation fold
        X_val_enc[encoded_col] = X_val_fold[col].map(smoothed).fillna(global_mean)
        
        # Add to training fold (using same mapping for consistency)
        X_train_enc[encoded_col] = X_train_fold[col].map(smoothed).fillna(global_mean)
    
    return X_train_enc, X_val_enc

## 5. Cross-Validation Function with Target Encoding

In [72]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_with_target_encoding(model, model_name, X, y, encode_cols, n_folds=5):
    """
    Evaluate model using KFold with target encoding inside the loop.
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = []
    predictions = np.zeros(len(y))
    global_mean = y.mean()
    
    print(f"\n{model_name} - {n_folds}-fold CV with target encoding for: {encode_cols}")
    print("-" * 60)
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train_fold = X.iloc[train_idx].copy()
        X_val_fold = X.iloc[val_idx].copy()
        y_train_fold = y[train_idx]
        y_val_fold = y[val_idx]
        
        # Apply target encoding
        X_train_enc, X_val_enc = apply_target_encoding(
            X_train_fold, y_train_fold, X_val_fold, 
            encode_cols, global_mean, smoothing=20
        )
        
        # Drop original encoded columns (now replaced by encoded versions)
        X_train_enc = X_train_enc.drop(columns=encode_cols)
        X_val_enc = X_val_enc.drop(columns=encode_cols)
        
        # Train and predict
        model_clone = model.__class__(**model.get_params())
        model_clone.fit(X_train_enc, y_train_fold)
        y_pred = model_clone.predict(X_val_enc)
        predictions[val_idx] = y_pred
        
        score = rmse(y_val_fold, y_pred)
        cv_scores.append(score)
        print(f"  Fold {fold+1}: RMSE = {score:.5f}")
    
    mean_score = np.mean(cv_scores)
    std_score = np.std(cv_scores)
    print(f"\n  Mean RMSE: {mean_score:.5f} (+/- {std_score:.5f})")
    
    return predictions, cv_scores

In [73]:
# Quick alpha tuning for Ridge
alphas = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0]
ridge_cv_scores = []

print("Tuning Ridge alpha...")
for alpha in alphas:
    ridge = Ridge(alpha=alpha, random_state=RANDOM_STATE)
    _, scores = evaluate_with_target_encoding(ridge, f"Ridge (alpha={alpha})", 
                                                X_train, y_train, target_encode_cols, n_folds=3)  # Quick 3-fold for tuning
    ridge_cv_scores.append(np.mean(scores))

best_alpha = alphas[np.argmin(ridge_cv_scores)]
print(f"\nBest alpha: {best_alpha}")

# Final Ridge with best alpha
ridge_final = Ridge(alpha=best_alpha, random_state=RANDOM_STATE)
ridge_preds, ridge_scores = evaluate_with_target_encoding(
    ridge_final, f"Ridge (alpha={best_alpha})", 
    X_train, y_train, target_encode_cols, n_folds=5
)

Tuning Ridge alpha...

Ridge (alpha=0.01) - 3-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
------------------------------------------------------------
  Fold 1: RMSE = 0.16441
  Fold 2: RMSE = 0.17698
  Fold 3: RMSE = 0.15836

  Mean RMSE: 0.16658 (+/- 0.00775)

Ridge (alpha=0.1) - 3-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
------------------------------------------------------------
  Fold 1: RMSE = 0.16090
  Fold 2: RMSE = 0.17773
  Fold 3: RMSE = 0.14354

  Mean RMSE: 0.16072 (+/- 0.01396)

Ridge (alpha=0.5) - 3-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Exterior2nd']
------------------------------------------------------------
  Fold 1: RMSE = 0.14591
  Fold 2: RMSE = 0.17591
  Fold 3: RMSE = 0.12700

  Mean RMSE: 0.14960 (+/- 0.02014)

Ridge (alpha=1.0) - 3-fold CV with target encoding for: ['MSSubClass', 'Neighborhood', 'Exterior1st', 'Ext